# Overview

This project explores consumer behavior on a Brazilian e-commerce marketplace
using SQL (DuckDB) and Python. Working across 99,000+ orders, 3,000+ sellers,
and 70+ product categories, the analysis investigates what customers buy, what
drives satisfaction, how retention breaks down, and what separates high-performing
sellers from the rest.

**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Tools:** DuckDB, Python, Pandas, Google Colab  
**Author:** Gawa Léo Sunang

## Key Findings
1. Health & beauty and bed & bath dominate by order volume; watches & gifts leads by average transaction value
2. Price does not predict satisfaction. review scores are flat across all price tiers
3. Delivery timing is the strongest satisfaction driver. late orders score 2.57 vs 4.29 for early ones
4. 97% of customers never return after their first order, acquisition vastly outpaces retention
5. High-revenue sellers follow two distinct strategies; revenue rank alone is a poor proxy for seller quality

## Table of Contents
1. Setup
2. Data Exploration
3. Data Cleaning
4. Analysis
  - What customers actually buy
  - Does price affect satisfaction?
  - Does delivery speed affect satisfaction?
  - Repeat purchase behavior
  - What traits do top revenue sellers share?



# Setup


In [39]:
from google.colab import files
uploaded = files.upload()

Saving olist_customers_dataset.csv to olist_customers_dataset (2).csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset (2).csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset (2).csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset (2).csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset (2).csv
Saving olist_orders_dataset.csv to olist_orders_dataset (2).csv
Saving olist_products_dataset.csv to olist_products_dataset (2).csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset (2).csv
Saving product_category_name_translation.csv to product_category_name_translation (2).csv


In [40]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute("""
    CREATE TABLE orders AS SELECT * FROM read_csv_auto('/content/olist_orders_dataset.csv');
    CREATE TABLE order_items AS SELECT * FROM read_csv_auto('/content/olist_order_items_dataset.csv');
    CREATE TABLE customers AS SELECT * FROM read_csv_auto('/content/olist_customers_dataset.csv');
    CREATE TABLE products AS SELECT * FROM read_csv_auto('/content/olist_products_dataset.csv');
    CREATE TABLE sellers AS SELECT * FROM read_csv_auto('/content/olist_sellers_dataset.csv');
    CREATE TABLE reviews AS SELECT * FROM read_csv_auto('/content/olist_order_reviews_dataset.csv');
    CREATE TABLE payments AS SELECT * FROM read_csv_auto('/content/olist_order_payments_dataset.csv');
    CREATE TABLE category_translation AS SELECT * FROM read_csv_auto('/content/product_category_name_translation.csv');
""")

print("All tables loaded.")

All tables loaded.


# Data Exploration

In [41]:
tables = ['orders', 'order_items', 'customers', 'products',
          'sellers', 'reviews', 'payments', 'category_translation']

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    print('='*50)
    schema = con.execute(f"DESCRIBE {table}").df()
    print(schema[['column_name', 'column_type']].to_string(index=False))


TABLE: orders
                  column_name column_type
                     order_id     VARCHAR
                  customer_id     VARCHAR
                 order_status     VARCHAR
     order_purchase_timestamp   TIMESTAMP
            order_approved_at   TIMESTAMP
 order_delivered_carrier_date   TIMESTAMP
order_delivered_customer_date   TIMESTAMP
order_estimated_delivery_date   TIMESTAMP

TABLE: order_items
        column_name column_type
           order_id     VARCHAR
      order_item_id      BIGINT
         product_id     VARCHAR
          seller_id     VARCHAR
shipping_limit_date   TIMESTAMP
              price      DOUBLE
      freight_value      DOUBLE

TABLE: customers
             column_name column_type
             customer_id     VARCHAR
      customer_unique_id     VARCHAR
customer_zip_code_prefix     VARCHAR
           customer_city     VARCHAR
          customer_state     VARCHAR

TABLE: products
               column_name column_type
                product_id     VARC

In [42]:
for table in tables:
    print(f"\n{'='*50}")
    print(f"NULL CHECK: {table}")
    print('='*50)

    # Get column names
    cols = con.execute(f"DESCRIBE {table}").df()['column_name'].tolist()

    # Build null count query dynamically
    null_counts = ', '.join([f"COUNT(*) - COUNT({c}) AS {c}" for c in cols])
    query = f"SELECT COUNT(*) AS total_rows, {null_counts} FROM {table}"

    result = con.execute(query).df()
    total = result['total_rows'][0]
    print(f"Total rows: {total}")
    print("\nNull counts per column:")
    for col in cols:
        null_val = result[col][0]
        pct = round(null_val / total * 100, 1)
        if null_val > 0:
            print(f"{col}: {null_val} nulls ({pct}%)")
        else:
            print(f"  ✓  {col}: no nulls")


NULL CHECK: orders
Total rows: 99441

Null counts per column:
  ✓  order_id: no nulls
  ✓  customer_id: no nulls
  ✓  order_status: no nulls
  ✓  order_purchase_timestamp: no nulls
order_approved_at: 160 nulls (0.2%)
order_delivered_carrier_date: 1783 nulls (1.8%)
order_delivered_customer_date: 2965 nulls (3.0%)
  ✓  order_estimated_delivery_date: no nulls

NULL CHECK: order_items
Total rows: 112650

Null counts per column:
  ✓  order_id: no nulls
  ✓  order_item_id: no nulls
  ✓  product_id: no nulls
  ✓  seller_id: no nulls
  ✓  shipping_limit_date: no nulls
  ✓  price: no nulls
  ✓  freight_value: no nulls

NULL CHECK: customers
Total rows: 99441

Null counts per column:
  ✓  customer_id: no nulls
  ✓  customer_unique_id: no nulls
  ✓  customer_zip_code_prefix: no nulls
  ✓  customer_city: no nulls
  ✓  customer_state: no nulls

NULL CHECK: products
Total rows: 32951

Null counts per column:
  ✓  product_id: no nulls
product_category_name: 610 nulls (1.9%)
product_name_lenght: 610 

# Data cleaning

In [43]:
# Orders: remove undelivered orders (nulls in delivery dates are undelivered)
con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT * FROM orders
    WHERE order_status = 'delivered'
    AND order_delivered_customer_date IS NOT NULL
""")

# Products: remove rows with no category name
con.execute("""
    CREATE OR REPLACE TABLE products AS
    SELECT * FROM products
    WHERE product_category_name IS NOT NULL
""")

# Reviews: coalesce null comment fields to empty string
con.execute("""
    CREATE OR REPLACE TABLE reviews AS
    SELECT
        review_id,
        order_id,
        review_score,
        COALESCE(review_comment_title, '') AS review_comment_title,
        COALESCE(review_comment_message, '') AS review_comment_message,
        review_creation_date,
        review_answer_timestamp
    FROM reviews
""")

print("Cleaning complete.")

Cleaning complete.


In [44]:
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,} rows")

orders: 96,470 rows
order_items: 112,650 rows
customers: 99,441 rows
products: 32,341 rows
sellers: 3,095 rows
reviews: 99,224 rows
payments: 103,886 rows
category_translation: 71 rows


#Analysis

##What customers actually buy.
Top 15 categories ranked by total revenue with order volume and average item price to distinguish high-frequency from high-value segments.

In [45]:
con.execute("""
    SELECT
        ct.product_category_name_english AS category,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        ROUND(AVG(oi.price), 2) AS avg_item_price
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 15
""").df()

,category,total_orders,total_revenue,avg_item_price
0,health_beauty,8647,1233131.72,130.28
1,watches_gifts,5493,1165898.98,199.06
2,bed_bath_table,9272,1023434.76,93.44
3,sports_leisure,7529,954673.55,113.25
4,computers_accessories,6529,888613.62,116.27
5,furniture_decor,6307,711927.69,87.25
6,housewares,5743,615628.69,90.60
7,cool_stuff,3559,610204.10,164.12
8,auto,3809,578849.35,139.85
9,toys,3803,471097.49,116.93


Health & beauty leads by order volume (8,647 orders) while watches & gifts leads by average item price (R$199). This split is meaningful: high-frequency, lower-ticket categories like health & beauty and bed & bath drive volume, while lower-frequency categories like watches and office furniture drive value per transaction.

## Does price affect satisfaction?
Orders segmented into four price tiers to test whether higher prices correlate with higher satisfaction.

In [46]:
con.execute("""
    WITH order_avg_price AS (
        SELECT
            o.order_id,
            AVG(oi.price) AS avg_price
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY o.order_id
    ),
    price_tiered AS (
        SELECT
            order_id,
            avg_price,
            CASE
                WHEN avg_price < 50 THEN '1. Under R$50'
                WHEN avg_price < 150 THEN '2. R$50–150'
                WHEN avg_price < 300 THEN '3. R$150–300'
                ELSE '4. Over R$300'
            END AS price_tier
        FROM order_avg_price
    )
    SELECT
        pt.price_tier,
        COUNT(*) AS total_orders,
        ROUND(AVG(r.review_score), 3) AS avg_review_score
    FROM price_tiered pt
    JOIN reviews r ON pt.order_id = r.order_id
    GROUP BY pt.price_tier
    ORDER BY pt.price_tier
""").df()

,price_tier,total_orders,avg_review_score
0,1. Under R$50,31736,4.170
1,2. R$50–150,44205,4.149
2,3. R$150–300,14058,4.149
3,4. Over R$300,6354,4.151


Review scores are remarkably stable across all price tiers, ranging only from 4.149 to 4.170. Price alone does not predict satisfaction on this platform. This suggests that other factors like delivery speed, product accuracy, seller reliability are stronger drivers of customer experience than what was paid. For a marketplace, improving logistics and seller quality likely has more impact on satisfaction than pricing strategy.

##Does delivery speed affect satisfaction?
Orders classified as early or late relative to the estimated delivery date, compared against average review score.

In [47]:
con.execute("""
    WITH delivery_performance AS (
        SELECT
            o.order_id,
            CASE
                WHEN o.order_delivered_customer_date < o.order_estimated_delivery_date
                    THEN '1. Early'
                WHEN o.order_delivered_customer_date = o.order_estimated_delivery_date
                    THEN '2. On time'
                ELSE '3. Late'
            END AS delivery_status,
            DATEDIFF('day', o.order_estimated_delivery_date, o.order_delivered_customer_date)
                AS days_from_estimate
        FROM orders o
        WHERE o.order_delivered_customer_date IS NOT NULL
        AND o.order_estimated_delivery_date IS NOT NULL
    )
    SELECT
        dp.delivery_status,
        COUNT(*) AS total_orders,
        ROUND(AVG(r.review_score), 3) AS avg_review_score,
        ROUND(AVG(dp.days_from_estimate), 1) AS avg_days_from_estimate
    FROM delivery_performance dp
    JOIN reviews r ON dp.order_id = r.order_id
    GROUP BY dp.delivery_status
    ORDER BY dp.delivery_status
""").df()

,delivery_status,total_orders,avg_review_score,avg_days_from_estimate
0,1. Early,88653,4.294,-13.7
1,3. Late,7700,2.566,8.8


Delivery timing is the strongest satisfaction signal in the dataset. Orders arriving early (the vast majority) average a review score of 4.29, while late orders drop sharply to 2.57, a difference of 1.7 points on a 5-point scale. The volume asymmetry is worth noting: roughly 92% of delivered orders arrived early, suggesting the platform deliberately sets conservative delivery estimates to underpromise and overdeliver. The 8% of late orders however disproportionately damage platform reputation, making late delivery prevention a high-leverage operational problem. Combined with Q2, a clear picture emerges: customers don't differentiate much by price, but they care deeply about receiving their order when expected.

##Repeat purchase behavior
Customers grouped by total order count to measure retention, with top 5 purchased categories per segment.

In [48]:
con.execute("""
    WITH customer_orders AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS total_orders
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    )
    SELECT
        CASE
            WHEN total_orders = 1 THEN '1. One-time'
            WHEN total_orders = 2 THEN '2. Two orders'
            WHEN total_orders = 3 THEN '3. Three orders'
            ELSE '4. Four or more'
        END AS customer_segment,
        COUNT(*) AS total_customers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_customers
    FROM customer_orders
    GROUP BY customer_segment
    ORDER BY customer_segment
""").df()

,customer_segment,total_customers,pct_of_customers
0,1. One-time,90549,97.0
1,2. Two orders,2573,2.8
2,3. Three orders,181,0.2
3,4. Four or more,47,0.1


In [49]:
con.execute("""
    WITH customer_order_count AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS total_orders
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    ),
    customer_segment AS (
        SELECT
            customer_unique_id,
            CASE
                WHEN total_orders = 1 THEN '1. One-time'
                WHEN total_orders = 2 THEN '2. Two orders'
                WHEN total_orders = 3 THEN '3. Three orders'
                ELSE '4. Four or more'
            END AS segment
        FROM customer_order_count
    ),
    category_by_segment AS (
        SELECT
            cs.segment,
            ct.product_category_name_english AS category,
            COUNT(*) AS total_purchases,
            ROW_NUMBER() OVER (
                PARTITION BY cs.segment
                ORDER BY COUNT(*) DESC
            ) AS category_rank
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN customer_segment cs ON c.customer_unique_id = cs.customer_unique_id
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        JOIN category_translation ct ON p.product_category_name = ct.product_category_name
        GROUP BY cs.segment, category
    )
    SELECT segment, category, total_purchases
    FROM category_by_segment
    WHERE category_rank <= 5
    ORDER BY segment, category_rank
""").df()

,segment,category,total_purchases
0,1. One-time,bed_bath_table,9880
1,1. One-time,health_beauty,8936
2,1. One-time,sports_leisure,7790
3,1. One-time,furniture_decor,7362
4,1. One-time,computers_accessories,7115
5,2. Two orders,bed_bath_table,910
6,2. Two orders,furniture_decor,734
7,2. Two orders,sports_leisure,548
8,2. Two orders,health_beauty,470
9,2. Two orders,computers_accessories,455


Repeat purchasing is rare: 97% of customers place only one order, with just 3% returning for a second or more. This is a significant retention challenge, the platform is far better at acquisition than at building loyalty. Category preferences are largely consistent across segments, suggesting repeat buyers aren't a distinct customer type with different tastes. The one exception is the "four or more" segment, where fashion bags & accessories and watches & gifts enter the top 5, hinting that the platform's most loyal customers skew toward higher-value, style-driven purchases. For a second-hand marketplace where repeat purchasing is the core business model, this retention gap would be a critical area of focus.

## Sellers that drive the most value
top 20 sellers by revenue profiled across order volume, average price, review score, on-time delivery rate, and category breadth.

In [50]:
con.execute("""
    WITH seller_stats AS (
        SELECT
            oi.seller_id,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(oi.price), 2) AS total_revenue,
            ROUND(AVG(oi.price), 2) AS avg_item_price,
            ROUND(AVG(r.review_score), 3) AS avg_review_score,
            ROUND(AVG(CASE
                WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
                THEN 1.0 ELSE 0.0
            END) * 100, 1) AS on_time_pct,
            COUNT(DISTINCT p.product_category_name) AS distinct_categories
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN reviews r ON o.order_id = r.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE o.order_delivered_customer_date IS NOT NULL
        GROUP BY oi.seller_id
        HAVING COUNT(DISTINCT o.order_id) >= 50
    )
    SELECT *
    FROM seller_stats
    ORDER BY total_revenue DESC
    LIMIT 20
""").df()

,seller_id,total_orders,total_revenue,avg_item_price,avg_review_score,on_time_pct,distinct_categories
0,4869f7a5dfa277a7dca6462dcf3b52b2,1116,225586.34,197.88,4.139,88.5,10
1,53243585a1d6dc2643021fd1853d8905,346,215904.44,542.47,4.128,96.2,2
2,4a3ca9315b744ce9f8e9374361493884,1753,197225.32,101.19,3.829,89.0,7
3,fa1c13f2614d7b5c4749cbc52fecda94,574,189649.54,329.83,4.374,90.3,5
4,7c67e1448b00f6e969d365cea6b010ab,967,186664.01,137.46,3.354,90.5,6
5,7e93a43ef30c4f03f38b393420bc753a,318,165751.50,516.36,4.364,94.7,7
6,da8622b14eb17ae2831f4ac5b9dab84a,1305,161574.27,103.24,4.075,93.0,4
7,7a67c85e85bb2ce8582c35f2203ad736,1137,139188.73,120.93,4.268,94.2,2
8,1025f0e2d44d7041d6cf58b6550e0bfa,902,138691.40,97.53,3.868,90.9,4
9,955fee9216a65b617aa5c0531780ce60,1253,130823.82,89.36,4.089,92.0,22


- Top-revenue sellers follow two distinct strategies: high volume at low average
prices, or low volume at high average prices. Neither path guarantees quality.
but the data suggests high-ticket sellers tend to score better on satisfaction
and delivery reliability, possibly because the economics of expensive items
incentivize more careful fulfillment.

- The strongest signal is at the extremes. The seller with the highest review score
in this group (4.45) operates in a single product category with a high average
price suggesting that specialization and focus may be a stronger predictor of
customer experience than scale. Meanwhile, the worst-reviewed seller in the group
(3.35) combines high volume with a low average price and below-average on-time
delivery, a profile that generates revenue while quietly eroding platform trust.

- For a marketplace, this distinction matters: not all revenue is equal. A seller
generating 186k with a 3.35 review score is a different kind of asset than one
generating 165k with a 4.36 score. Identifying and differentiating between
these profiles is a prerequisite for any serious seller quality programme.